# Project 1 - Attitude Constraints

This walkthrough is the teaching document for the constrained one-axis attitude project. The implementation details stay in project-local modules, but the model, limits, experiment sequence, and interpretation are visible here.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys
import numpy as np
from IPython.display import FileLink, display

repo_root = Path.cwd()
while not (repo_root / "projects").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from projects.project_1_attitude_constraints import config, scenario
from projects.project_1_attitude_constraints.baseline_control import simulate_lqr, simulate_saturated_lqr
from projects.project_1_attitude_constraints.animation import save_attitude_replay_html
from projects.project_1_attitude_constraints.plots import (
    plot_constraint_activity,
    plot_phase_plane,
    plot_terminal_geometry,
    plot_time_histories,
)

output_root = Path(os.environ.get("THIMPC_OUTPUT_DIR", "/tmp/thimpc_walkthroughs"))
project_output = output_root / "project_1_attitude_constraints"
np.set_printoptions(precision=3, suppress=True)


## Main Experiment Parameters

These are the values used in the cells below. Students should not need to open `config.py` just to understand the experiment.


In [ ]:
dt = config.DT
horizon = config.HORIZON
n_steps = int(os.environ.get("THIMPC_PROJECT1_STEPS", "40"))
initial_state = config.X0.copy()
angle_limits = (config.X_MIN[0], config.X_MAX[0])
rate_limits = (config.X_MIN[1], config.X_MAX[1])
input_limits = (config.U_MIN[0], config.U_MAX[0])
rate_bound = config.RATE_BOUND.copy()
soft_constraint_penalty = 5_000.0

print(f"dt = {dt}, horizon = {horizon}, simulated steps = {n_steps}")
print("initial state [theta, omega] =", initial_state)
print("angle limits [rad] =", angle_limits)
print("rate limits [rad/s] =", rate_limits)
print("input limits =", input_limits)
print("input-rate bound per step =", rate_bound)


## Model And Constraints

We use a sampled one-axis attitude double integrator:

`theta[k+1] = theta[k] + dt omega[k] + 0.5 dt^2 u[k]`

`omega[k+1] = omega[k] + dt u[k]`

Physical interpretation:
- `theta` is the attitude angle error;
- `omega` is the angular-rate error;
- `u` is the available torque command.


In [ ]:
A = config.A
B = config.B
Q = config.Q
R = config.R

print("A =\n", A)
print("B =\n", B)
print("Q =\n", Q)
print("R =\n", R)


## LQR And Saturated LQR

What are we comparing?
- LQR is the unconstrained design;
- saturated LQR clips only the current torque command.

Control message: clipping respects the actuator at this instant, but it does not make the closed-loop controller constrained optimal.


In [ ]:
K, P = scenario.lqr_gain(A, B, Q, R)
X_lqr, U_lqr = simulate_lqr(A, B, K, initial_state, n_steps)
X_sat, U_sat = simulate_saturated_lqr(A, B, K, initial_state, n_steps, config.U_MIN, config.U_MAX)

print("LQR gain K =", K)
print("max |u_LQR| =", float(np.max(np.abs(U_lqr))))
print("max |u_saturated| =", float(np.max(np.abs(U_sat))))


## Constrained MPC

Now the controller predicts the attitude and rate over a horizon and solves for a sequence of torques. The CasADi construction stays in `scenario.py`; the closed-loop experiment remains visible here.


In [ ]:
mpc_run = scenario.simulate_receding_mpc(initial_state, n_steps)
rate_limited_mpc_run = scenario.simulate_receding_mpc(initial_state, n_steps, rate_bound=rate_bound)

runs = {
    "LQR": (X_lqr, U_lqr),
    "saturated LQR": (X_sat, U_sat),
    "MPC": (mpc_run["X"], mpc_run["U"]),
    "MPC + rate limit": (rate_limited_mpc_run["X"], rate_limited_mpc_run["U"]),
}

time = np.arange(n_steps + 1) * dt
plot_time_histories(project_output / "figures" / "time_histories.png", time, runs, config.X_MIN, config.X_MAX, config.U_MIN, config.U_MAX)
plot_phase_plane(project_output / "figures" / "phase_plane.png", runs, config.X_MIN, config.X_MAX)


## Replay: Attitude Motion Under Constraints

Run this cell after the simulation to generate a replay.
The replay is saved under outputs/ and is not committed.
Open the generated HTML file in a browser.


In [ ]:
replay_output = repo_root / "outputs" / "project_1_attitude_constraints"
replay_output.mkdir(parents=True, exist_ok=True)
replay_path = replay_output / "attitude_replay.html"

animation_stride = max(1, n_steps // 24)
save_attitude_replay_html(
    replay_path,
    time=time,
    saturated_lqr=(X_sat, U_sat),
    mpc=(mpc_run["X"], mpc_run["U"]),
    stride=animation_stride,
    interval=80,
)

print(f"Replay saved to: {replay_path}")
display(FileLink(replay_path))


### Interpretation

- LQR is useful as a reference, but it can ask for unavailable torque.
- Saturated LQR respects the torque limit by clipping locally.
- MPC uses the prediction horizon to keep the state and input limits in the optimization problem.
- The rate-limited MPC shows the extra cost of commanding smoother torque changes.


## Constraint Activity And Rate Limits

What are we checking?
- distance to the angle limits;
- distance to the torque limits;
- whether soft slack was used.


In [ ]:
plot_constraint_activity(
    project_output / "figures" / "constraint_activity.png",
    time,
    rate_limited_mpc_run["X"],
    rate_limited_mpc_run["U"],
    rate_limited_mpc_run["slack"],
    config.X_MIN,
    config.X_MAX,
    config.U_MIN,
    config.U_MAX,
)

for label, (X, U) in runs.items():
    theta_violations = scenario.count_violations(X[:, 0], config.X_MIN[0], config.X_MAX[0])
    input_violations = scenario.count_violations(U[:, 0], config.U_MIN[0], config.U_MAX[0])
    final_norm = np.linalg.norm(X[-1])
    print(f"{label:16s} theta violations = {theta_violations:2d}, input violations = {input_violations:2d}, final ||x|| = {final_norm:.3f}")


### Interpretation

- A zero margin means the corresponding constraint is active.
- Input-rate limits can delay the response even when the torque itself is feasible.
- Solver statuses should be checked whenever the problem is close to the boundary.


## Infeasibility And Soft Constraints

Hard constraints can make a problem infeasible. A soft constraint does not remove the limit from the teaching story; it adds a priced violation so the optimizer can recover and report how much violation was needed.


In [ ]:
infeasible_initial_state = np.array([1.5, 0.0])
tight_angle_min = np.array([-0.8, -2.5])
tight_angle_max = np.array([0.8, 2.5])

_, hard_info = scenario.solve_attitude_mpc(
    infeasible_initial_state,
    x_min=tight_angle_min,
    x_max=tight_angle_max,
)
_, soft_info = scenario.solve_attitude_mpc(
    infeasible_initial_state,
    x_min=tight_angle_min,
    x_max=tight_angle_max,
    soft_theta=True,
    slack_weight=soft_constraint_penalty,
)

print("hard-constrained status:", hard_info.status)
print("soft-constrained status:", soft_info.status)
if soft_info.success and soft_info.slack.size:
    print("initial theta slack:", float(soft_info.slack[0, 0]))


### Interpretation

- The hard problem says, honestly, that the initial angle is already outside the allowed set.
- The soft problem gives the controller a controlled way to continue.
- Slack is not free; the penalty should be large enough that violations happen only when needed.


## Terminal Geometry

The terminal cost from the DARE acts like a local value function. The ellipses below help connect the algebraic matrix `P` to the phase-plane picture.


In [ ]:
plot_terminal_geometry(project_output / "figures" / "terminal_geometry.png", P, config.X_MIN, config.X_MAX)


## Modification Cell

Change one value in this cell and rerun the MPC cells above.

Suggested experiments:
- decrease `modified_rate_bound` to make smoother torque mandatory;
- increase or decrease `modified_slack_penalty` to change how reluctant the soft constraint is;
- change `modified_initial_state` to start closer to the angle limit.


In [ ]:
# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.
